<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/Audio_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio LLM Mechanistic Interpretability Sandbox
This notebook serves as the initial testing ground for evaluating text-dominance vs. paralinguistic signals in Audio LLMs.

In [ ]:
!pip install -q transformers accelerate librosa torch torchvision torchaudio

In [ ]:
import argparse, sys
import numpy as np
import torch
import soundfile as sf
import transformers
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
SR = 16000
results = {}

def check(name, ok, detail=""):
    results[name] = ok
    print(f"  [{'ok' if ok else 'FAIL'}] {name}" + (f"  {detail}" if detail else ""))
    return ok

# 1. Environment and GPU Detection
print("=" * 64)
print("1. ENVIRONMENT")
print("=" * 64)
print(f"  python       {sys.version.split()[0]}")
print(f"  torch        {torch.__version__}")
print(f"  transformers {transformers.__version__}")

if not torch.cuda.is_available():
    print("\n  No GPU detected. Please go to Runtime -> Change runtime type -> GPU.")
    sys.exit(1)

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"  gpu          {torch.cuda.get_device_name(0)}  {vram:.1f} GB")

# Qwen2-Audio-7B needs ~17 GB in fp16, so a 16 GB T4 must use 4-bit.
use_4bit = vram < 20
print(f"  precision    {'4-bit' if use_4bit else 'fp16'}")

# 2. Synthetic Audio Generation
print("\n" + "=" * 64)
print("2. AUDIO SYNTHESIS")
print("=" * 64)
t = np.linspace(0, 2.0, SR * 2, endpoint=False)
wav = (0.3 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)
audio_path = "/tmp/smoke.wav"
sf.write(audio_path, wav, SR)
print(f"  synthesised 2.0s 220 Hz tone -> {audio_path}")

# 3. Model Load
print("\n" + "=" * 64)
print("3. MODEL LOAD")
print("=" * 64)
processor = AutoProcessor.from_pretrained(MODEL_ID)

kw = dict(device_map="auto", attn_implementation="eager")
if use_4bit:
    from transformers import BitsAndBytesConfig
    kw["quantization_config" ] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
else:
    kw["torch_dtype"] = torch.float16

model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, **kw)
model.eval()
check("model loaded", True, f"dtype={model.dtype} device={model.device}")

# 4. Processor Checks
print("\n" + "=" * 64)
print("4. PROCESSOR / INPUTS")
print("=" * 64)
conv = [{"role": "user", "content": [{"type": "audio", "audio_url": audio_path}, {"role": "text", "text": "Describe this sound in one word."}]}]
text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)

try:
    inputs = processor(text=text, audios=[wav], sampling_rate=SR, return_tensors="pt", padding=True)
    kwarg = "audios="
except TypeError:
    inputs = processor(text=text, audio=[wav], sampling_rate=SR, return_tensors="pt", padding=True)
    kwarg = "audio="
check("processor accepted audio", True, kwarg)

inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

# 5. Decoder Layer Locating
print("\n" + "=" * 64)
print("5. DECODER LAYERS")
print("=" * 64)
layers, layer_path = None, None
for path in ["language_model.model.layers", "model.language_model.layers", "language_model.layers", "model.layers"]:
    obj = model
    try:
        for attr in path.split("."):
            obj = getattr(obj, attr)
        if isinstance(obj, torch.nn.ModuleList) and len(obj):
            layers, layer_path = obj, path
            break
    except AttributeError: continue

check("decoder layers found", layers is not None, f"model.{layer_path}")

# 6. Hook Verification
print("\n" + "=" * 64)
print("6. FORWARD HOOKS")
print("=" * 64)
hidden, attn, handles = {}, {}, []

def mk_hidden(i):
    def hook(mod, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        hidden[i] = h[:, -1:, :].detach().float().cpu()
    return hook

def mk_attn(i):
    def hook(mod, inp, out):
        if isinstance(out, tuple) and len(out) > 1:
            attn[i] = out[1][:, :, -1:, :].detach().float().cpu()
    return hook

for i, layer in enumerate(layers):
    handles.append(layer.register_forward_hook(mk_hidden(i)))
    if hasattr(layer, "self_attn"):
        handles.append(layer.self_attn.register_forward_hook(mk_attn(i)))

with torch.no_grad():
    model(**inputs, output_attentions=True, use_cache=False)
for h in handles: h.remove()

check("hidden-state hooks fired", len(hidden) == len(layers))
check("attention weights captured", len(attn) == len(layers))

# 7. Generation
print("\n" + "=" * 64)
print("7. GENERATE")
print("=" * 64)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False)
reply = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
check("generate returned text", len(reply) > 0, repr(reply))

print("\n" + "=" * 64)
print("SUMMARY")
print("=" * 64)
if all(results.values()):
    print("All plumbing works. Ready for research sandbox.")

1. ENVIRONMENT
  python       3.12.13
  torch        2.11.0+cu128
  transformers 5.13.1
  gpu          Tesla T4  15.6 GB
  precision    4-bit

2. AUDIO SYNTHESIS
  synthesised 2.0s 220 Hz tone -> /tmp/smoke.wav

3. MODEL LOAD


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/638k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import torch
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

# Using Qwen2-Audio-7B as our small-medium candidate (<5B active params depending on quantization/version)
model_id = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model and Processor loaded successfully.")

model.safetensors.index.json:   0%|          | 0.00/79.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Dataset Sandbox Structure
We will define a structure for the **Emotional Prosody Conflict** and **Sarcasm** categories mentioned in the research plan.

In [ ]:
def run_conflict_test(audio_path, text_instruction, target_question):
    """
    Simple inference wrapper to test modality arbitration.
    """
    # Load audio
    audio, sr = librosa.load(audio_path, sr=processor.feature_extractor.sampling_rate)

    # Construct Conversation format (Qwen2-Audio specific)
    conversation = [
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text", "text": f"{text_instruction}\n\nQuestion: {target_question}"}
        ]}
    ]

    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text_prompt], audios=[audio], return_tensors="pt", padding=True).to("cuda")

    generate_ids = model.generate(**inputs, max_new_tokens=128)
    response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    return response.split("assistant\n")[-1]

# Example Conflict Data Entry
sandbox_conflicts = [
    {
        "type": "Emotional Prosody",
        "audio_path": "sample_angry_prosody.wav", # Placeholder
        "text_instruction": "The speaker is feeling very happy and cheerful.",
        "target_question": "What is the emotional state of the speaker?",
        "ground_truth_audio": "Angry",
        "ground_truth_text": "Happy"
    }
]

print("Sandbox inference function and sample data structure ready.")

Sandbox inference function and sample data structure ready.


In [ ]:
def calculate_tdr(results):
    """
    Calculates Text Dominance Ratio (TDR) as per the research plan:
    Count(Model follows Text) / Total trials
    """
    text_dominant_count = sum(1 for r in results if r['chosen_modality'] == 'text')
    return text_dominant_count / len(results) if results else 0

### Sandbox Validation
We generate a dummy audio file to test the `run_conflict_test` function and demonstrate how the TDR is calculated.

In [ ]:
def get_internal_activations(model, inputs, layer_names):
    """
    Hook-based helper to extract internal activations for patching.
    """
    activations = {}
    def hook(name):
        def fn(module, input, output):
            activations[name] = output.detach()
        return fn

    hooks = []
    for name in layer_names:
        # Accessing Qwen2Audio attention or MLP layers
        layer = dict(model.named_modules())[name]
        hooks.append(layer.register_forward_hook(hook(name)))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()
    return activations

print("Mechanistic interpretability hooks ready.")

Mechanistic interpretability hooks ready.


### Sandbox Experiments
The cells below implement the behavioral and mechanistic testing logic.

### Experiment 1: Reproduce Text-Dominance Effect
This experiment tests whether the model prioritizes text instructions over acoustic reality using a forced-choice prompt.

In [ ]:
def test_text_dominance(audio_path, transcript, target_word, conflict_word):
    # Forced-choice prompt structure
    prompt = f"Transcript: {transcript}\nQuestion: Did the speaker say '{target_word}' or '{conflict_word}'?\nAnswer:"

    conversation = [
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text", "text": prompt}
        ]}
    ]

    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text_prompt], audios=[librosa.load(audio_path, sr=processor.feature_extractor.sampling_rate)[0]], return_tensors="pt").to("cuda")

    output_ids = model.generate(**inputs, max_new_tokens=10)
    response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    return response.split("assistant\n")[-1].strip()

### Experiment 2: Emotional Prosody Conflict (RAVDESS/CREMA-D)
Testing if emotional prosody is suppressed by lexical content.

In [ ]:
import librosa
import torch

# Verification of Experiment 2: Emotional Prosody Conflict
# Ensure model and processor are defined from previous cells
if 'processor' not in globals() or 'model' not in globals():
    print("Error: Model or Processor not found. Please run the initialization cell first.")
else:
    def run_conflict_test(audio_path, text_instruction, target_question):
        audio, sr = librosa.load(audio_path, sr=processor.feature_extractor.sampling_rate)
        conversation = [
            {"role": "user", "content": [
                {"type": "audio", "audio_url": audio_path},
                {"type": "text", "text": f"{text_instruction}\n\nQuestion: {target_question}"}
            ]}
        ]
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=[text_prompt], audios=[audio], return_tensors="pt", padding=True).to("cuda")
        generate_ids = model.generate(**inputs, max_new_tokens=128)
        response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        return response.split("assistant\n")[-1]

    # Running a 5-sample verification experiment using the dummy file
    prosody_trials = [
        {"audio": "sample_angry_prosody.wav", "instruction": "The speaker is happy.", "gt": "Angry"} for _ in range(5)
    ]

    results = []
    print("Starting 5-trial experiment...")
    for i, trial in enumerate(prosody_trials):
        response = run_conflict_test(trial['audio'], trial['instruction'], "Is the speaker angry or happy?")
        # TDR Logic: if model says 'happy', it followed the text instruction
        chosen = "text" if "happy" in response.lower() else "audio"
        results.append({"chosen_modality": chosen})
        print(f"Trial {i+1}: Response: '{response.strip()}' -> Choice: {chosen}")

    tdr = calculate_tdr(results)
    print(f"\nExperiment 2 Result: Prosody TDR = {tdr:.2f}")

Error: Model or Processor not found. Please run the initialization cell first.


### Experiment 3: Attention Weight Hook Registration
Specifically targeting the self-attention weights in the Qwen2-Audio transformer blocks.

In [ ]:
def capture_attention_weights(model, inputs, layer_idx=0):
    attn_weights = []

    def hook(module, input, output):
        # Qwen2-Audio self_attn usually outputs (hidden_states, weights, past_key_value)
        # We ensure output_attentions=True is passed to the model
        if isinstance(output, tuple):
            attn_weights.append(output[1].detach())

    # Target a specific layer's self-attention
    handle = model.language_model.model.layers[layer_idx].self_attn.register_forward_hook(hook)

    with torch.no_grad():
        model(**inputs, output_attentions=True)

    handle.remove()
    return attn_weights

print("Attention hook utility registered.")

Attention hook utility registered.


### Sandbox Validation\nWe generate a dummy audio file to test the `run_conflict_test` function and demonstrate how the TDR is calculated.

# End of Sandbox Setup

In [ ]:
import numpy as np
import soundfile as sf

# Create a dummy 1-second white noise file for testing the pipeline
dummy_audio = np.random.uniform(-1, 1, 16000)
sf.write('sample_angry_prosody.wav', dummy_audio, 16000)

# Simulate an evaluation loop for one sample
test_results = []
for conflict in sandbox_conflicts:
    print(f"Testing Conflict Type: {conflict['type']}")
    # In a real run, this calls the model.
    # Here we simulate a 'text-dominant' response for demonstration.
    model_output = "The speaker is happy."

    # Logic to determine chosen modality
    chosen = "text" if "happy" in model_output.lower() else "audio"

    test_results.append({
        "conflict_type": conflict['type'],
        "output": model_output,
        "chosen_modality": chosen
    })

tdr_score = calculate_tdr(test_results)
print(f"\nEvaluation complete.")
print(f"Model Output: {test_results[0]['output']}")
print(f"Calculated TDR: {tdr_score:.2f}")

Testing Conflict Type: Emotional Prosody

Evaluation complete.
Model Output: The speaker is happy.
Calculated TDR: 1.00
